# BrainBoost Iteration 3: Data Science Notebook

**Project:** BrainBoost  
**Iteration:** Iteration 3  
**Focus:** Smart Reminder logic, wearable health data integration, data quality, feature engineering, and evaluation.

This notebook adds a data science layer to the BrainBoost project. It demonstrates how manual habit data and optional wearable data can be transformed into useful features for personalised reminders and student brain-health insights.

The notebook uses a small synthetic dataset so the workflow can be demonstrated without exposing real user data.

## 1. Data Science Objective

BrainBoost currently supports onboarding, habit tracking, progress feedback, mini games, article recommendations, smart reminders, and planned wearable integration.

From a data science perspective, Iteration 3 should answer three questions:

1. How can raw student lifestyle data be converted into meaningful analytical features?
2. How can those features trigger personalised, explainable reminders?
3. How can the system handle missing, duplicated, or mixed-source data from manual check-ins and wearable devices?

## 1.1 How This Notebook Maps to the BrainBoost Web App

This notebook is designed to support the actual BrainBoost web application, not a separate project.

| Web App Area | Current / Planned Data | Notebook Section |
|---|---|---|
| `/onboarding` | Saves `brainboostSnapshot` with overall score and domain scores | Data dictionary, scoring context, ethics |
| `/dashboard` | Shows brain-health snapshot, domain priorities, insights, and article picks | Feature engineering and EDA charts |
| `/habits` | Stores daily `sleep_hours`, `screen_time`, `physical_activity`, and `date` | Synthetic habit dataset and cleaning |
| Guest mode | Uses `localStorage` key `bb_guest_habits` | App-compatible export table |
| Signed-in mode | Sends habit records to `/api/habits` | App-compatible export table |
| `/progress` | Uses habit history, streaks, and game score trends | Weekly summary, game-score analysis, and evaluation metrics |
| `/games` | Stores mini-game results in `game_scores` through `/api/games` | Cognitive-game performance analysis |
| `/articles` | Recommends content from lower-scoring domains | Explainability and reason text |
| Iteration 3 Smart Reminder | Uses recent habit patterns and user timing/tone settings | Rule engine and validation tests |
| Iteration 3 Wearable Integration | Adds optional sleep/activity data with permission | Wearable/manual fallback logic |

The notebook starts with the exact fields used by your current frontend and backend tables, then derives extra analytical columns for scoring, reminders, progress, and reporting.

In [ ]:
# If running locally and packages are missing, install them first:
# pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## 2. Backend-Aligned Habit Dataset

The dataset below mirrors the current BrainBoost backend table used by `server/routes/habits.js`.

Current backend table shown in your database: `habits(id, user_id, sleep_hours, screen_time, physical_activity, date, created_at)`.

Because I do not have authenticated access to your production database inside this notebook, the first dataset is a backend-shaped sample. It is not random unrelated data: it uses the same column names, value bands, duplicate rule, timestamps, and payload shape as `/api/habits`. If you export real rows from the backend later, they can replace `backend_habits_sample` without changing the rest of the notebook.

A helper script has been added at `server/scripts/export-brainboost-data.mjs`. When `DATABASE_URL` is available, run it from the `server` folder to create `brainboost_habits_export.json` and `brainboost_game_scores_export.json` for this notebook.

In [ ]:
np.random.seed(42)

# These are the exact value bands used by HabitTracker.jsx and server/routes/habits.js.
SLEEP_BANDS = ['< 6', '6', '7', '8', '10+']
SCREEN_BANDS = ['< 2h', '2-4h', '4-6h', '6-8h', '8h+']

# The current backend has no public unauthenticated export endpoint. For report/demo use,
# this sample preserves the exact backend schema and allowed values.
user_ids = ['guest_demo_001', 'guest_demo_002', 'user_demo_003', 'user_demo_004', 'user_demo_005']
dates = pd.date_range('2026-05-01', periods=14, freq='D')

rows = []
record_id = 1
for user_id in user_ids:
    sleep_profile = np.random.choice(['low', 'mixed', 'healthy'], p=[0.30, 0.45, 0.25])
    screen_profile = np.random.choice(['low', 'mixed', 'high'], p=[0.25, 0.45, 0.30])
    activity_profile = np.random.choice(['low', 'mixed', 'high'], p=[0.25, 0.45, 0.30])

    for date in dates:
        if sleep_profile == 'low':
            sleep_hours = np.random.choice(['< 6', '6', '7'], p=[0.45, 0.40, 0.15])
        elif sleep_profile == 'healthy':
            sleep_hours = np.random.choice(['7', '8', '10+'], p=[0.35, 0.55, 0.10])
        else:
            sleep_hours = np.random.choice(SLEEP_BANDS, p=[0.15, 0.25, 0.30, 0.25, 0.05])

        if screen_profile == 'high':
            screen_time = np.random.choice(['4-6h', '6-8h', '8h+'], p=[0.25, 0.45, 0.30])
        elif screen_profile == 'low':
            screen_time = np.random.choice(['< 2h', '2-4h', '4-6h'], p=[0.45, 0.40, 0.15])
        else:
            screen_time = np.random.choice(SCREEN_BANDS, p=[0.15, 0.30, 0.30, 0.18, 0.07])

        if activity_profile == 'high':
            physical_activity = bool(np.random.choice([True, False], p=[0.80, 0.20]))
        elif activity_profile == 'low':
            physical_activity = bool(np.random.choice([True, False], p=[0.30, 0.70]))
        else:
            physical_activity = bool(np.random.choice([True, False], p=[0.55, 0.45]))

        rows.append({
            'id': record_id,
            'user_id': user_id,
            'sleep_hours': sleep_hours,
            'screen_time': screen_time,
            'physical_activity': physical_activity,
            'date': date.strftime('%Y-%m-%d'),
            'created_at': (date + pd.Timedelta(hours=np.random.randint(0, 24), minutes=np.random.randint(0, 60))).strftime('%Y-%m-%d %H:%M:%S')
        })
        record_id += 1

backend_habits_sample = pd.DataFrame(rows)

# Optional: if you export real rows from your backend as JSON, place the file next to this notebook
# as brainboost_habits_export.json. The notebook will use that instead of the demo sample.
export_path = Path('brainboost_habits_export.json')
if export_path.exists():
    df_raw = pd.read_json(export_path)
    print('Loaded real backend export:', export_path)
else:
    df_raw = backend_habits_sample.copy()
    print('Using backend-shaped demo data. Replace with brainboost_habits_export.json for real backend rows.')

display(df_raw.head(12))
print('Columns:', list(df_raw.columns))

## 3. Data Dictionary

This table is based on the actual backend comment in `server/routes/habits.js`, not a separate invented schema.

| Field | Backend Type | Allowed / Example Values | Used By | Privacy Consideration |
|---|---|---|---|---|
| id | Auto-generated ID | 1, 2, 3 | Database record identity | Internal only |
| user_id | TEXT | Clerk user ID or `guest_` prefixed guest ID | `/api/habits`, `/api/habits/streak`, Progress page | Pseudonymous identifier; should not expose real student ID |
| sleep_hours | TEXT | `< 6`, `6`, `7`, `8`, `9+`, `10+` | HabitTracker, Dashboard score, Progress charts, Smart Reminder | Health-related lifestyle data |
| screen_time | TEXT | `< 2h`, `2-4h`, `4-6h`, `6-8h`, `8h+` | HabitTracker, Dashboard score, Smart Reminder | Behavioural data |
| physical_activity | BOOLEAN | `true`, `false` | HabitTracker, Dashboard score, Progress timeline | Health/activity data |
| date | DATE | `YYYY-MM-DD` | One record per user per day, streak calculation | Low risk alone, higher risk when linked to habits |
| created_at | TIMESTAMP | `2026-05-15 00:14:22` | Audit/insert timestamp shown in database | Can reveal usage timing patterns |

Iteration 3 wearable data should be treated as an extension that maps into these existing fields, not as a replacement for the current backend.

The screenshots also show the current `game_scores` backend table:

| Field | Backend Type | Example Values | Used By | Privacy Consideration |
|---|---|---|---|---|
| id | SERIAL | 19, 20, 21 | Database record identity | Internal only |
| user_id | TEXT | Clerk user ID | `/api/games`, `/api/games/leaderboard/:gameId`, Progress page | Pseudonymous identifier |
| game_id | TEXT | `reaction`, `memory`, `visual_pattern`, `stroop`, `mental_math` | MiniGames and Progress | Performance category |
| score | INTEGER | 291, 15, 6 | Leaderboards, achievements, progress charts | Cognitive-game performance data |
| metadata | JSONB | `{"time_seconds":29}`, `{"rounds":[...]}` | Game-specific details | Can reveal detailed performance patterns |
| played_at | TIMESTAMP | `2026-05-14 23:38:08` | Progress ordering and recent history | Can reveal usage timing patterns |
| display_name | TEXT | `Sameer #3546`, `Yuhan #9240` | Leaderboard display | Should avoid real full names if public |

## 4. Backend Validation and Cleaning

The backend accepts `sleep_hours`, `screen_time`, `physical_activity`, and `date`. Before using records for analytics or reminders, the notebook validates that values match the same bands used by the web app.

- `sleep_hours` must be one of the app's sleep bands
- `screen_time` must be one of the app's screen-time bands
- `physical_activity` must be boolean
- `date` must parse as a date
- `(user_id, date)` should be unique, matching the POST duplicate guard in `/api/habits`

This keeps the data science layer combined with the current backend rather than inventing a new storage model.

In [ ]:
df = df_raw.copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')

valid_sleep = set(SLEEP_BANDS + ['9+'])  # '9+' is supported by scoring.js as a legacy saved value.
valid_screen = set(SCREEN_BANDS)

df['valid_sleep_hours'] = df['sleep_hours'].isin(valid_sleep)
df['valid_screen_time'] = df['screen_time'].isin(valid_screen)
df['valid_physical_activity'] = df['physical_activity'].isin([True, False])
df['valid_date'] = df['date'].notna()
df['duplicate_user_date'] = df.duplicated(['user_id', 'date'], keep=False)

validation_summary = pd.DataFrame({
    'check': ['sleep_hours band', 'screen_time band', 'physical_activity boolean', 'date parsed', 'duplicate user/date'],
    'failed_records': [
        (~df['valid_sleep_hours']).sum(),
        (~df['valid_screen_time']).sum(),
        (~df['valid_physical_activity']).sum(),
        (~df['valid_date']).sum(),
        df['duplicate_user_date'].sum()
    ]
})

display(validation_summary)
display(df.head(12))

## 4.1 Backend Payload and SQL Shape

Your current BrainBoost app sends daily habit records to the backend with these fields:

```json
{
  "sleep_hours": 7,
  "screen_time": 4,
  "physical_activity": true,
  "date": "2026-05-01"
}
```

The same shape is used when guest data is stored in `bb_guest_habits` and when signed-in users post to `/api/habits`. The SQL insert in your backend is:

```sql
INSERT INTO habits (user_id, sleep_hours, screen_time, physical_activity, date)
VALUES ($1, $2, $3, $4, $5)
RETURNING *;
```

So this notebook keeps those fields as the main dataset and only derives extra analytical columns after loading.

## 4.3 Iteration 3 Wearable Integration Without Breaking the Backend

The current backend does not yet store `steps`, `active_minutes`, or raw wearable sleep values. To keep Iteration 3 realistic, wearable data should be mapped into the existing `habits` table first, then optionally stored with extra metadata.

Recommended backend approach:

1. Keep `/api/habits` as the main habit endpoint.
2. Convert wearable sleep duration into the existing `sleep_hours` bands.
3. Convert wearable movement data into the existing `physical_activity` boolean.
4. Add optional source metadata only if the team has time.

Possible future SQL migration:

```sql
ALTER TABLE habits
ADD COLUMN IF NOT EXISTS data_source TEXT DEFAULT 'manual',
ADD COLUMN IF NOT EXISTS wearable_metadata JSONB DEFAULT '{}'::jsonb;
```

Example wearable mapping:

| Wearable Input | Existing Backend Field | Mapping Rule |
|---|---|---|
| 5.4 hours sleep | `sleep_hours` | `< 6` |
| 7.6 hours sleep | `sleep_hours` | `8` |
| 9.2 hours sleep | `sleep_hours` | `10+` |
| 25 active minutes or 5000+ steps | `physical_activity` | `true` |
| Below activity threshold | `physical_activity` | `false` |

This keeps the data science notebook combined with your backend instead of creating a separate wearable-only schema.

In [ ]:
brainboost_habit_payload = df.assign(
    date=df['date'].dt.strftime('%Y-%m-%d')
)[['user_id', 'date', 'sleep_hours', 'screen_time', 'physical_activity']]

display(brainboost_habit_payload.head(10))

# Example payload that matches the existing /api/habits POST body.
example_api_payload = brainboost_habit_payload.drop(columns=['user_id']).iloc[0].to_dict()
example_api_payload

## 4.4 Backend-Aligned Game Scores

Your database screenshot also shows `game_scores`, which is used by the MiniGames and Progress pages. This is valuable for data science because it gives a second behavioural signal: cognitive-game performance over time.

The notebook loads `brainboost_game_scores_export.json` if available. Otherwise, it uses a small backend-shaped demo table matching your schema.

In [ ]:
game_export_path = Path('brainboost_game_scores_export.json')

if game_export_path.exists():
    game_scores = pd.read_json(game_export_path)
    print('Loaded real game_scores backend export:', game_export_path)
else:
    game_scores = pd.DataFrame([
        {'id': 19, 'user_id': 'user_demo_005', 'game_id': 'memory', 'score': 15, 'metadata': {'time_seconds': 29}, 'played_at': '2026-05-13 18:44:15', 'display_name': 'Sameer #3546'},
        {'id': 20, 'user_id': 'user_demo_004', 'game_id': 'reaction', 'score': 291, 'metadata': {'rounds': [303, 301, 282]}, 'played_at': '2026-05-13 23:38:08', 'display_name': 'Yuhan #9240'},
        {'id': 21, 'user_id': 'user_demo_004', 'game_id': 'memory', 'score': 26, 'metadata': {'time_seconds': 48}, 'played_at': '2026-05-14 01:39:52', 'display_name': 'Yuhan #9240'},
        {'id': 22, 'user_id': 'user_demo_004', 'game_id': 'memory', 'score': 21, 'metadata': {'time_seconds': 40}, 'played_at': '2026-05-14 01:41:21', 'display_name': 'Yuhan #9240'},
        {'id': 23, 'user_id': 'user_demo_004', 'game_id': 'reaction', 'score': 333, 'metadata': {'rounds': [268, 280, 449]}, 'played_at': '2026-05-14 03:01:30', 'display_name': 'Yuhan #9240'},
        {'id': 24, 'user_id': 'user_demo_004', 'game_id': 'reaction', 'score': 307, 'metadata': {'rounds': [360, 324, 282]}, 'played_at': '2026-05-14 22:26:35', 'display_name': 'Yuhan #7119'},
        {'id': 25, 'user_id': 'user_demo_005', 'game_id': 'memory', 'score': 22, 'metadata': {'time_seconds': 65}, 'played_at': '2026-05-14 23:16:33', 'display_name': 'Sameer #1536'},
        {'id': 26, 'user_id': 'user_demo_005', 'game_id': 'visual_pattern', 'score': 5, 'metadata': {'max_level': 5}, 'played_at': '2026-05-14 23:17:59', 'display_name': 'Sameer #1536'},
        {'id': 27, 'user_id': 'guest_demo_002', 'game_id': 'visual_pattern', 'score': 6, 'metadata': {'max_level': 6}, 'played_at': '2026-05-15 00:13:57', 'display_name': 'Shivani #6080'},
    ])
    print('Using backend-shaped game_scores demo data. Replace with brainboost_game_scores_export.json for real backend rows.')

game_scores['played_at'] = pd.to_datetime(game_scores['played_at'], errors='coerce')
display(game_scores)

## 5. Data Quality Checks

Before using data for reminders, the app should check data completeness and quality. This matters because incomplete or biased data can produce poor recommendations.

In [ ]:
quality_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(1)
})

backend_value_counts = {
    'sleep_hours': df['sleep_hours'].value_counts().sort_index(),
    'screen_time': df['screen_time'].value_counts().sort_index(),
    'physical_activity': df['physical_activity'].value_counts().sort_index()
}

display(quality_summary)
for field, counts in backend_value_counts.items():
    print(f'\n{field}')
    display(counts)

## 6. Feature Engineering for Smart Reminders

The following features convert raw daily records into reminder-ready signals.

| Feature | Derived From Backend Field | Meaning | Used For |
|---|---|---|
| sleep_score | `sleep_hours` | Same score mapping as `src/utils/scoring.js` | Dashboard-style analysis |
| screen_score | `screen_time` | Same reverse score mapping as `src/utils/scoring.js` | Dashboard-style analysis |
| activity_score | `physical_activity` | Same boolean score mapping as `src/utils/scoring.js` | Dashboard-style analysis |
| low_sleep_3d | `sleep_score` | Repeated weak sleep score across recent check-ins | Sleep reset reminder |
| high_screen_3d | `screen_score` | Repeated weak screen-time score across recent check-ins | Wind-down reminder |
| low_activity_3d | `activity_score` | Repeated low activity across recent check-ins | Movement reminder |
| risk_score | All three backend habit fields | Combined behavioural risk score | Prioritise reminder type |

In [ ]:
df = df.sort_values(['user_id', 'date']).reset_index(drop=True)

# These mappings intentionally mirror src/utils/scoring.js.
sleep_score_map = {'< 6': 20, '6': 40, '7': 60, '8': 100, '9+': 80, '10+': 80}
screen_score_map = {'< 2h': 100, '2-4h': 80, '4-6h': 60, '6-8h': 40, '8h+': 20}

df['sleep_score'] = df['sleep_hours'].map(sleep_score_map).fillna(50)
df['screen_score'] = df['screen_time'].map(screen_score_map).fillna(50)
df['activity_score'] = np.where(df['physical_activity'], 100, 40)

grouped = df.groupby('user_id', group_keys=False)
df['sleep_score_avg_3d'] = grouped['sleep_score'].rolling(3, min_periods=2).mean().reset_index(level=0, drop=True)
df['screen_score_avg_3d'] = grouped['screen_score'].rolling(3, min_periods=2).mean().reset_index(level=0, drop=True)
df['activity_score_avg_3d'] = grouped['activity_score'].rolling(3, min_periods=2).mean().reset_index(level=0, drop=True)

df['low_sleep_3d'] = df['sleep_score_avg_3d'] < 60
df['high_screen_3d'] = df['screen_score_avg_3d'] < 60
df['low_activity_3d'] = df['activity_score_avg_3d'] < 60

df['risk_score'] = (
    df['low_sleep_3d'].astype(int) * 3 +
    df['high_screen_3d'].astype(int) * 2 +
    df['low_activity_3d'].astype(int) * 2
)

df[['user_id', 'date', 'sleep_hours', 'screen_time', 'physical_activity', 'sleep_score_avg_3d', 'screen_score_avg_3d', 'activity_score_avg_3d', 'low_sleep_3d', 'high_screen_3d', 'low_activity_3d', 'risk_score']].head(15)

## 7. Explainable Smart Reminder Rule Engine

For an early-stage student prototype, an explainable rule-based approach is appropriate. It is easier to test, easier to explain to users, and safer than a black-box model for wellbeing-related suggestions.

The rules below match the Iteration 3 user stories:

| Condition | Reminder Type | Explanation |
|---|---|---|
| `sleep_score_avg_3d < 60` | Sleep reset | Recent backend sleep bands are weak |
| `screen_score_avg_3d < 60` | Wind-down | Recent backend screen-time bands are high |
| `activity_score_avg_3d < 60` | Movement break | Recent backend activity values show repeated rest days |
| No risk condition | No warning reminder | Avoids unnecessary notification fatigue |

In [ ]:
def choose_reminder(row, tone='direct'):
    """Return an explainable reminder type and message for one daily record."""
    if row['low_sleep_3d']:
        reminder_type = 'sleep_reset'
        reason = f"Your recent sleep score average is {row['sleep_score_avg_3d']:.0f}/100 based on backend sleep_hours values."
    elif row['high_screen_3d']:
        reminder_type = 'wind_down'
        reason = f"Your recent screen-time score average is {row['screen_score_avg_3d']:.0f}/100 based on backend screen_time values."
    elif row['low_activity_3d']:
        reminder_type = 'movement_break'
        reason = f"Your recent activity score average is {row['activity_score_avg_3d']:.0f}/100 based on backend physical_activity values."
    else:
        return 'none', 'No reminder needed', 'No unhealthy pattern was detected from recent backend habit records.'

    messages = {
        "direct": {
            "sleep_reset": "Your sleep has been below target recently. Try an earlier reset tonight to support focus tomorrow.",
            "wind_down": "Your evening screen time has been high. Try a short wind-down before sleep.",
            "movement_break": "Your movement has been low recently. A 10-minute walk could help reset your focus.",
            "study_support": "Study Crunch Mode is active. Protect sleep, movement, hydration, and short breaks.",
            "data_check_in": "Some sleep data is missing. Add a quick check-in so BrainBoost can give better insights."
        },
        "chill": {
            "sleep_reset": "Sleep has been a bit low lately. Maybe take it easy tonight and aim for an earlier reset.",
            "wind_down": "Your screen time is running late. A small wind-down could make tomorrow easier.",
            "movement_break": "You've been sitting a lot lately. A short walk might help your brain breathe a little.",
            "study_support": "Crunch week is on. Keep the basics steady: sleep, water, movement, breaks.",
            "data_check_in": "A few sleep entries are missing. Quick check-in when you have a moment?"
        },
        "hype": {
            "sleep_reset": "Brain recharge needed. Aim for stronger sleep tonight so tomorrow's focus has backup.",
            "wind_down": "Screen-time boss fight detected. Wind down now and protect tomorrow's focus.",
            "movement_break": "Movement reset time. Ten minutes now can help your next study block hit better.",
            "study_support": "Study Crunch Mode is on. Lock in the basics: sleep, water, movement, repeat.",
            "data_check_in": "BrainBoost needs a little data fuel. Add your sleep check-in to sharpen insights."
        }
    }

    return reminder_type, messages[tone][reminder_type], reason

reminder_outputs = df.apply(lambda row: choose_reminder(row, tone='direct'), axis=1, result_type='expand')
df[['reminder_type', 'reminder_message', 'reminder_reason']] = reminder_outputs

df[['user_id', 'date', 'sleep_hours', 'screen_time', 'physical_activity', 'risk_score', 'reminder_type', 'reminder_reason', 'reminder_message']].head(20)

## 8. Validation Test Cases

These tests show that the smart reminder logic behaves according to the acceptance criteria in the Iteration 3 report.

In [ ]:
test_cases = pd.DataFrame([
    {
        "case": "Low sleep for 3 days",
        "sleep_score_avg_3d": 40,
        "screen_score_avg_3d": 80,
        "activity_score_avg_3d": 100,
        "low_sleep_3d": True,
        "high_screen_3d": False,
        "low_activity_3d": False,
        "expected": "sleep_reset"
    },
    {
        "case": "High evening screen use",
        "sleep_score_avg_3d": 80,
        "screen_score_avg_3d": 40,
        "activity_score_avg_3d": 100,
        "low_sleep_3d": False,
        "high_screen_3d": True,
        "low_activity_3d": False,
        "expected": "wind_down"
    },
    {
        "case": "Healthy pattern",
        "sleep_score_avg_3d": 80,
        "screen_score_avg_3d": 80,
        "activity_score_avg_3d": 100,
        "low_sleep_3d": False,
        "high_screen_3d": False,
        "low_activity_3d": False,
        "expected": "none"
    }
])

actual = []
for _, row in test_cases.iterrows():
    reminder_type, _, _ = choose_reminder(row)
    actual.append(reminder_type)

test_cases["actual"] = actual
test_cases["passed"] = test_cases["expected"] == test_cases["actual"]

display(test_cases[["case", "expected", "actual", "passed"]])
assert test_cases["passed"].all(), "One or more smart reminder validation tests failed."

## 9. Exploratory Data Analysis

The charts below show how BrainBoost can turn habit data into insight. In the final app, similar summaries could support the dashboard, progress page, or iteration report.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sleep_order = ['< 6', '6', '7', '8', '9+', '10+']
sns.countplot(data=df, x='sleep_hours', order=[x for x in sleep_order if x in df['sleep_hours'].unique()], ax=axes[0], color='#2563eb')
axes[0].set_title('Backend sleep_hours Distribution')
axes[0].set_xlabel('sleep_hours band')

screen_order = ['< 2h', '2-4h', '4-6h', '6-8h', '8h+']
sns.countplot(data=df, x='screen_time', order=screen_order, ax=axes[1], color='#f59e0b')
axes[1].set_title('Backend screen_time Distribution')
axes[1].set_xlabel('screen_time band')
axes[1].tick_params(axis='x', rotation=30)

reminder_counts = df['reminder_type'].value_counts().reset_index()
reminder_counts.columns = ['reminder_type', 'count']
sns.barplot(data=reminder_counts, x='count', y='reminder_type', ax=axes[2], color='#16a34a')
axes[2].set_title('Reminder Types Triggered from Backend Fields')
axes[2].set_xlabel('Number of daily records')
axes[2].set_ylabel('Reminder type')

plt.tight_layout()
plt.show()

In [ ]:
weekly_summary = (
    df.assign(week=df['date'].dt.isocalendar().week)
      .groupby(['user_id', 'week'])
      .agg(
          avg_sleep_score=('sleep_score', 'mean'),
          avg_screen_score=('screen_score', 'mean'),
          avg_activity_score=('activity_score', 'mean'),
          checkins=('id', 'count'),
          reminders_triggered=('reminder_type', lambda x: (x != 'none').sum())
      )
      .round(2)
      .reset_index()
)

weekly_summary

## 9.1 Backend Game Score Analysis

The `game_scores` table supports the MiniGames and Progress pages. The score direction depends on the game:

- `reaction`: lower score is better because it measures milliseconds
- `memory`: lower score is better because it measures moves
- `stroop`, `visual_pattern`, `mental_math`: higher score is better

This section creates a backend-aligned game summary that could support Progress-page analytics or a final report chart.

In [ ]:
lower_is_better = {'reaction', 'memory'}

game_summary_rows = []
for game_id, game_df in game_scores.groupby('game_id'):
    if game_id in lower_is_better:
        best_score = game_df['score'].min()
    else:
        best_score = game_df['score'].max()
    game_summary_rows.append({
        'game_id': game_id,
        'plays': len(game_df),
        'unique_users': game_df['user_id'].nunique(),
        'best_score': best_score,
        'score_direction': 'lower is better' if game_id in lower_is_better else 'higher is better'
    })

game_summary = pd.DataFrame(game_summary_rows).sort_values('game_id')
display(game_summary)

plt.figure(figsize=(9, 4))
sns.countplot(data=game_scores, x='game_id', color='#2563eb')
plt.title('Mini-game Plays from Backend game_scores')
plt.xlabel('game_id')
plt.ylabel('Number of plays')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 9.2 How These Outputs Can Appear in the Web App

These notebook outputs can be translated directly into BrainBoost screens:

| Notebook Output | Web App Placement | User-Facing Benefit |
|---|---|---|
| `sleep_score_avg_3d`, `screen_score_avg_3d`, `activity_score_avg_3d` | Dashboard or Smart Reminder settings | Shows why a reminder was triggered using backend habit values |
| `reminder_type` | Planned Smart Reminder feature | Selects the correct reminder category |
| `reminder_message` | Reminder notification or in-app reminder card | Gives short student-centred advice |
| `reminder_reason` | Dashboard explanation text | Makes recommendations transparent |
| `brainboost_habit_payload` | `/habits`, `bb_guest_habits`, `/api/habits` | Matches existing app data structure |
| weekly summary | `/progress` page | Shows habit trends over time |
| `game_summary` | `/progress` and `/games` pages | Summarises mini-game plays, best scores, and score direction |
| proposed `data_source` field | Planned Wearable Integration screen | Could distinguish manual vs wearable data in a future backend migration |

This means the notebook is not just analysis for the report. It is a prototype of the data logic that can support your React pages and Express API.

## 10. Evaluation Metrics for Iteration 3

The project can be evaluated using both data quality and engagement metrics.

| Goal | Metric | Why It Matters |
|---|---|---|
| Improve data completeness | Percent of days with sleep/activity data | More complete data improves insights |
| Reduce manual effort | Percent of records with future `data_source = wearable` | Shows wearable integration value after backend migration |
| Avoid notification fatigue | Number of reminders per user per week | Too many reminders can reduce trust |
| Improve engagement | Habit check-in completion rate | Shows users continue using the app |
| Improve explainability | Percent of reminders with a visible reason | Users should know why a reminder appears |
| Protect user autonomy | Reminder opt-out / tone change rate | Shows whether users feel in control |

In [ ]:
evaluation_metrics = {
    'backend_records_valid_percent': round((df[['valid_sleep_hours', 'valid_screen_time', 'valid_physical_activity', 'valid_date']].all(axis=1)).mean() * 100, 1),
    'duplicate_user_date_records': int(df['duplicate_user_date'].sum()),
    'records_with_reminder_percent': round((df['reminder_type'] != 'none').mean() * 100, 1),
    'average_reminders_per_user': round((df['reminder_type'] != 'none').groupby(df['user_id']).sum().mean(), 2),
    'reminders_with_reason_percent': round(df.loc[df['reminder_type'] != 'none', 'reminder_reason'].notna().mean() * 100, 1),
    'average_checkins_per_user': round(df.groupby('user_id')['id'].count().mean(), 2)
}

pd.DataFrame(evaluation_metrics.items(), columns=["metric", "value"])

## 11. Ethics, Privacy, and Bias Considerations

Because BrainBoost uses health-adjacent lifestyle data, the data science process must be transparent and careful.

**Key considerations:**

- Wearable users may not represent all students. Students without Apple Watch or Apple Health access should not receive a worse experience.
- Self-reported data may be inaccurate due to memory errors or social desirability bias.
- Consumer wearable data is useful for lifestyle insights but should not be treated as clinical-grade measurement.
- Reminders should be explainable, non-clinical, and user-controllable.
- The app should collect the minimum data required for the feature.
- Users should be able to disconnect wearable access and continue with manual check-ins.
- BrainBoost should avoid making medical, psychological, or diagnostic claims.

## 12. Conclusion

This notebook demonstrates how BrainBoost Iteration 3 can be supported by a clear data science workflow:

1. Start from the current backend `habits` schema.
2. Validate backend values and the one-record-per-user-per-day rule.
3. Apply the same scoring logic used by `src/utils/scoring.js`.
4. Engineer rolling behavioural features from backend habit records.
5. Use explainable rules to trigger smart reminders.
6. Map future wearable data into the existing backend fields before adding optional metadata.
7. Monitor data quality, engagement, and ethical risks.

This strengthens the project by showing that BrainBoost is not only a React application, but also a data-informed wellbeing prototype with transparent logic, privacy awareness, and measurable evaluation criteria.